# SSH Web Tool - Jupyter Notebook 集成示例

本示例展示如何在 Jupyter Notebook 中使用 `ssh_web_tool` 包：
1. 导入并初始化 SSH 管理器
2. 启动 Web UI（在浏览器中观察终端）
3. 建立 SSH 连接
4. 发送命令并在 Web 终端中实时看到执行过程
5. 连续执行命令（同一 shell 进程，cd 等状态保持）

## 前置条件
- 已安装 `ssh_web_tool` 包（`pip install -e .`）
- 有一台可连接的 SSH 服务器
- 浏览器可访问 `http://127.0.0.1:8765`

## 1. 导入包并初始化管理器

创建 `SSHWebTool` 实例，启动 Web UI。
启动后在浏览器中打开 http://127.0.0.1:8765 即可观察终端。

In [ ]:
from ssh_web_tool import SSHWebTool

# 初始化管理器，启动 Web UI（默认端口 8765）
# web_ui=True 会在后台线程启动 FastAPI 服务器
tool = SSHWebTool(web_ui=True, host='127.0.0.1', port=8765)

print('SSH Web Tool 已初始化')
print('Web UI: http://127.0.0.1:8765')
print('在浏览器中打开上面的地址，可以观察所有 SSH 终端')

## 2. 配置 SSH 连接信息

请修改下面的连接信息为你的实际服务器。

In [ ]:
# SSH 连接配置（请修改为你的实际服务器）
SSH_HOST = '47.108.145.84'      # 服务器地址
SSH_PORT = 22                     # 端口
SSH_USER = 'root'                 # 用户名
SSH_PASS = 'wrz@1234'             # 密码

print(f'目标服务器: {SSH_USER}@{SSH_HOST}:{SSH_PORT}')

## 3. 建立 SSH 连接

调用 `tool.connect()` 建立 SSH 连接并启动交互式终端。
连接成功后，在 Web UI 中可以看到这个终端。

**关键点**：`connect()` 会创建一个交互式 shell 终端，后续通过 `run_command(inject=True)` 发送的命令会直接注入到这个终端中执行，在 Web UI 中可以实时看到命令输入和输出。

In [ ]:
# 建立 SSH 连接（会自动启动交互式终端）
session_id = tool.connect(
    host=SSH_HOST,
    username=SSH_USER,
    password=SSH_PASS,
    port=SSH_PORT,
    terminal_name='Jupyter终端'  # 终端名称，在 Web UI 中显示
)

print(f'SSH 连接成功！')
print(f'会话 ID: {session_id}')
print()
print('现在请在浏览器中打开 http://127.0.0.1:8765')
print('你应该能看到一个名为「Jupyter终端」的终端标签')
print('后续在 Notebook 中执行的命令都会实时显示在这个终端中')

## 4. 发送命令（注入模式，Web 终端可见）

使用 `tool.run_command(session_id, command, inject=True)` 发送命令。

**inject=True（默认）**：命令注入到交互式终端中执行
- 在 Web UI 终端中可以看到命令被输入和执行
- 同一 shell 进程，cd、环境变量等状态保持
- 支持 Python、MySQL、vim 等交互程序

**inject=False**：独立执行命令（不显示在终端中）
- 不占用交互式终端
- 每次执行都是独立的 shell 进程
- 适合不需要观察的批量命令

In [ ]:
# 发送第一个命令：查看系统信息
# inject=True 表示注入到交互式终端，Web UI 中可见
result = tool.run_command(session_id, 'uname -a && uptime', inject=True)

print('=== 命令执行结果 ===')
print(result['output'])
print(f'退出码: {result["exit_code"]}')
print()
print('注意：在 Web UI 终端中应该能看到这条命令被输入和执行的过程！')

In [ ]:
# 发送第二个命令：查看当前目录
result = tool.run_command(session_id, 'pwd && ls -la', inject=True)

print('=== 当前目录和文件 ===')
print(result['output'])

## 5. 连续执行命令（状态保持）

因为是同一个交互式 shell 进程，`cd` 等命令的状态会保持。
这是注入模式的核心优势之一。

In [ ]:
# 连续执行：切换目录后查看文件
tool.run_command(session_id, 'cd /tmp', inject=True)
result = tool.run_command(session_id, 'pwd', inject=True)
print(f'切换后目录: {result["output"].strip()}')  # 应该是 /tmp

# 再切回根目录
tool.run_command(session_id, 'cd /', inject=True)
result = tool.run_command(session_id, 'pwd', inject=True)
print(f'切回后目录: {result["output"].strip()}')  # 应该是 /

## 6. 进入交互程序（如 Python）

注入模式支持进入 Python、MySQL、Redis 等交互程序。
进入后，可以继续发送命令与交互程序交互。

**注意**：进入交互程序后，命令执行的等待时间可能会稍长，因为需要检测交互程序的提示符。

In [ ]:
# 进入 Python 交互模式
result = tool.run_command(session_id, 'python3', inject=True, timeout=10)
print('=== 进入 Python ===')
print(result['output'])
print()
print('在 Web UI 终端中应该能看到 Python 提示符 >>>')

In [ ]:
# 在 Python 中执行代码
result = tool.run_command(session_id, 'print("Hello from Jupyter!")', inject=True, timeout=10)
print('=== Python 执行结果 ===')
print(result['output'])

# 计算 1+1
result = tool.run_command(session_id, '1 + 1', inject=True, timeout=10)
print('=== 1 + 1 = ===')
print(result['output'])

In [ ]:
# 退出 Python，回到 shell
result = tool.run_command(session_id, 'exit()', inject=True, timeout=10)
print('=== 退出 Python ===')
print(result['output'])
print()
print('在 Web UI 终端中应该回到 shell 提示符')

## 7. 查看终端状态

可以检测当前终端处于什么状态（shell、Python、MySQL 等）。

In [ ]:
# 查看终端当前状态
state = tool.get_terminal_state(session_id)
print('=== 终端状态 ===')
print(f'提示符类型: {state.get("prompt_type", "unknown")}')
print(f'是否忙碌: {state.get("is_busy", False)}')
print(f'最后输出: {state.get("last_output", "")[:100]}')

## 8. 查看所有活跃会话

可以列出当前所有活跃的 SSH 会话。

In [ ]:
# 列出所有活跃会话
sessions = tool.list_sessions()
print(f'当前活跃会话数: {len(sessions)}')
print()
for s in sessions:
    print(f'  会话ID: {s["session_id"]}')
    print(f'  主机: {s["host"]}:{s["port"]}')
    print(f'  用户: {s["username"]}')
    print(f'  终端名: {s["terminal_name"]}')
    print(f'  已连接: {s["connected"]}')
    print(f'  有shell: {s["has_shell"]}')
    print()

## 9. 发送原始输入（特殊字符、控制序列）

使用 `tool.send_input()` 发送原始输入，不等待结果。
适合发送 Ctrl+C、方向键等控制字符。

In [ ]:
# 示例：发送 Ctrl+C 中断当前命令
# tool.send_input(session_id, '\x03')  # Ctrl+C

# 示例：发送方向键上（查看历史命令）
# tool.send_input(session_id, '\x1b[A')  # Up arrow

print('send_input() 可用于发送控制字符')
print('Ctrl+C = \x03')
print('Ctrl+D = \x04')
print('方向键上 = \x1b[A')
print('方向键下 = \x1b[B')

## 10. 清理：关闭连接

使用完毕后，关闭 SSH 连接。

**注意**：关闭后 Web UI 中的终端标签也会消失。

In [ ]:
# 关闭单个会话
# tool.close_session(session_id)
# print(f'会话 {session_id} 已关闭')

# 或者关闭所有连接并停止 Web UI
# tool.close_all()
# print('所有连接已关闭，Web UI 已停止')

print('取消注释上面的代码来关闭连接')
print('保持连接打开的话，可以继续在 Web UI 中操作终端')

## 总结

### 核心优势
1. **Notebook 中编程，Web UI 中观察** — 在 Jupyter 中写代码执行命令，在浏览器终端中实时看到执行过程
2. **同一 shell 进程** — 注入模式下所有命令在同一个 shell 中执行，cd、环境变量等状态保持
3. **支持交互程序** — 可以进入 Python、MySQL、Redis、vim 等交互程序并继续交互
4. **连接持久化** — Notebook 关闭后，SSH 连接仍然保持（由本地 Server 维护），重新打开 Notebook 可以复用
5. **多终端管理** — 可以创建多个终端，在 Web UI 中通过标签切换

### 常用 API 速查
| 方法 | 说明 |
|------|------|
| `SSHWebTool(web_ui=True)` | 初始化管理器，启动 Web UI |
| `tool.connect(host, user, pass, port)` | 建立 SSH 连接，返回 session_id |
| `tool.run_command(sid, cmd, inject=True)` | 执行命令，inject=True 时 Web 终端可见 |
| `tool.send_input(sid, data)` | 发送原始输入（控制字符等），不等待结果 |
| `tool.get_terminal_state(sid)` | 获取终端状态（提示符类型、是否忙碌） |
| `tool.list_sessions()` | 列出所有活跃会话 |
| `tool.close_session(sid)` | 关闭单个会话 |
| `tool.close_all()` | 关闭所有连接，停止 Web UI |

### 注意事项
- `run_command(inject=True)` 默认超时 30 秒，长时间命令请增加 `timeout` 参数
- 进入交互程序后，命令检测依赖提示符匹配，可能需要稍长等待时间
- 如果 Web UI 中看不到命令，检查是否使用了 `inject=True`（默认是 True）
- 数据文件 `data.json` 和日志保存在运行 Notebook 的工作目录中